
### CUSTOMERS

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import os
import sys
from typing import List
from pyspark.sql import DataFrame
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
class Transformations:
    def dedup(self,df:DataFrame,dedup_cols:List,cdc:str):
        df = df.withColumn('dedupKey',concat(*dedup_cols))
        df = df.withColumn('dedupCount',row_number()\
            .over(Window.partitionBy(col('dedupKey')).orderBy(desc(cdc))))
        df = df.filter(col('dedupCount')==1)
        df = df.drop(col('dedupKey'),col('dedupCount'))

        return df
    
    def process_timestamp(self,df):
        df = df.withColumn('process_timestamp',current_timestamp())
        return df
    
    def upsert(self,df,key_cols,table,cdc):
        dlt_obj = DeltaTable.forName(spark,f"pyspark_dbt.silver.{table}")
        merge_condition = ' AND '.join([f"s.{i} = t.{i}" for i in key_cols])
        dlt_obj.alias("t").merge(
            df.alias("s"),
            merge_condition,
        ).whenMatchedUpdateAll(condition=f"t.{cdc} < s.{cdc}").whenNotMatchedInsertAll().execute()
        return 1


In [0]:
custom_obj = Transformations()

In [0]:
df_customers = spark.read.table('pyspark_dbt.bronze.customers')

df_customers = df_customers.withColumn('email_domain',split(col("email"),'@')[1])

df_customers = df_customers.withColumn('phone_number',regexp_replace(col('phone_number'), '[^0-9]', ''))

df_customers =df_customers.withColumn('full_name',concat(col('first_name'),lit(' '),col('last_name'))).drop(col('first_name'),col('last_name'))

df_customers = custom_obj.dedup(df_customers,['customer_id'],'last_updated_timestamp')

df_customers = custom_obj.process_timestamp(df_customers)

if not spark.catalog.tableExists('pyspark_dbt.silver.customers'):
    df_customers.write.format('delta').mode('append').saveAsTable('pyspark_dbt.silver.customers')
else:
    custom_obj.upsert(df_customers,['customer_id'],'customers','last_updated_timestamp')


### Drivers

In [0]:
df_drivers = spark.read.table('pyspark_dbt.bronze.drivers')

df_drivers = df_drivers.withColumn('phone_number',regexp_replace(col('phone_number'), '[^0-9]', ''))
df_drivers =df_drivers.withColumn('full_name',concat(col('first_name'),lit(' '),col('last_name'))).drop(col('first_name'),col('last_name'))

df_drivers = custom_obj.dedup(df_drivers,['driver_id'],'last_updated_timestamp')

df_drivers = custom_obj.process_timestamp(df_drivers)


if not spark.catalog.tableExists('pyspark_dbt.silver.drivers'):
    df_drivers.write.format('delta').mode('append').saveAsTable('pyspark_dbt.silver.drivers')
else:
    custom_obj.upsert(df_drivers,['driver_id'],'drivers','last_updated_timestamp')

### Locations

In [0]:
df_locations = spark.read.table('pyspark_dbt.bronze.locations')

df_locations = custom_obj.dedup(df_locations,['location_id'],'last_updated_timestamp')

df_locations = custom_obj.process_timestamp(df_locations)


if not spark.catalog.tableExists('pyspark_dbt.silver.locations'):
    df_locations.write.format('delta').mode('append').saveAsTable('pyspark_dbt.silver.locations')
else:
    custom_obj.upsert(df_locations,['location_id'],'locations','last_updated_timestamp')

### Payments

In [0]:
df_payments = spark.read.table('pyspark_dbt.bronze.payments')


df_payments = df_payments.withColumn('online_payment_status',
                       when((col('payment_method')=='Card')&(col('payment_status')=='Success'),'online-success')
                        .when((col('payment_method')=='Card')&(col('payment_status')=='Failed'),'online-failure')
                         .when((col('payment_method')=='Card')&(col('payment_status')=='Pending'),'online-pending')
                         .otherwise('offline')
                       )
                       
df_payments = custom_obj.dedup(df_payments,['payment_id'],'last_updated_timestamp')

df_payments = custom_obj.process_timestamp(df_payments)


if not spark.catalog.tableExists('pyspark_dbt.silver.payments'):
    df_payments.write.format('delta').mode('append').saveAsTable('pyspark_dbt.silver.payments')
else:
    custom_obj.upsert(df_payments,['payment_id'],'payments','last_updated_timestamp')


### Vehicles

In [0]:
df_vehicles = spark.read.table('pyspark_dbt.bronze.vehicles')
df_vehicles = df_vehicles.withColumn('make',upper(col('make')))

df_vehicles = custom_obj.dedup(df_vehicles,['vehicle_id'],'last_updated_timestamp')

df_vehicles = custom_obj.process_timestamp(df_vehicles)


if not spark.catalog.tableExists('pyspark_dbt.silver.vehicles'):
    df_vehicles.write.format('delta').mode('append').saveAsTable('pyspark_dbt.silver.vehicles')
else:
    custom_obj.upsert(df_vehicles,['vehicle_id'],'vehicles','last_updated_timestamp')